# Convolutions from Scratch Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Pad an array

Start with the smallest primitive: a function that pads with zeros around an H x W array.

In [ ]:
```python

import numpy as np

def pad2d(x, p):

    if p == 0:

        return x

    h, w = x.shape[-2:]

    out = np.zeros(x.shape[:-2] + (h + 2 * p, w + 2 * p), dtype=x.dtype)

    out[..., p:p + h, p:p + w] = x

    return out

x = np.arange(9).reshape(3, 3)

print(x)

print()

print(pad2d(x, 1))

In [ ]:
```

The trailing-axes trick `x.shape[:-2]` means the same function works on `(H, W)`, `(C, H, W)`, or `(N, C, H, W)` without modification.

### Step 2: 2D convolution with nested loops

The reference implementation — slow, but unambiguous. This is what `torch.nn.functional.conv2d` does in principle.

In [ ]:
```python

def conv2d_naive(x, w, b=None, stride=1, padding=0):

    c_in, h, w_in = x.shape

    c_out, c_in_w, kh, kw = w.shape

    assert c_in == c_in_w

    x_pad = pad2d(x, padding)

    h_out = (h + 2 * padding - kh) // stride + 1

    w_out = (w_in + 2 * padding - kw) // stride + 1

    out = np.zeros((c_out, h_out, w_out), dtype=np.float32)

    for oc in range(c_out):

        for i in range(h_out):

            for j in range(w_out):

                hs = i * stride

                ws = j * stride

                patch = x_pad[:, hs:hs + kh, ws:ws + kw]

                out[oc, i, j] = np.sum(patch * w[oc])

        if b is not None:

            out[oc] += b[oc]

    return out

In [ ]:
```

Four nested loops (output channel, row, column, plus the implicit sum over C_in, kh, kw). This is the ground truth you will check every faster implementation against.

### Step 3: Verify with a hand-designed kernel

Build a vertical Sobel kernel, apply it to a synthetic step image, and watch the vertical edge light up.

In [ ]:
```python

def synthetic_step_image():

    img = np.zeros((1, 16, 16), dtype=np.float32)

    img[:, :, 8:] = 1.0

    return img

sobel_x = np.array([

    [[-1, 0, 1],

     [-2, 0, 2],

     [-1, 0, 1]]

], dtype=np.float32)[None]

x = synthetic_step_image()

y = conv2d_naive(x, sobel_x, padding=1)

print(y[0].round(1))

In [ ]:
```

Expect large positive values on column 7 (left-to-right brightness increase) and zeros everywhere else. That single print is your sanity check that the math is right.

### Step 4: im2col

Convert every kernel-sized window in the input into a column of a matrix. For `C_in=3, K=3`, each column is 27 numbers.

In [ ]:
```python

def im2col(x, kh, kw, stride=1, padding=0):

    c_in, h, w = x.shape

    x_pad = pad2d(x, padding)

    h_out = (h + 2 * padding - kh) // stride + 1

    w_out = (w + 2 * padding - kw) // stride + 1

    cols = np.zeros((c_in * kh * kw, h_out * w_out), dtype=x.dtype)

    col = 0

    for i in range(h_out):

        for j in range(w_out):

            hs = i * stride

            ws = j * stride

            patch = x_pad[:, hs:hs + kh, ws:ws + kw]

            cols[:, col] = patch.reshape(-1)

            col += 1

    return cols, h_out, w_out

In [ ]:
```

It is still a Python loop, but now the heavy lifting will be a single vectorised matmul.

### Step 5: Fast conv via im2col + matmul

Replace the quadruple loop with one matrix multiplication.

In [ ]:
```python

def conv2d_im2col(x, w, b=None, stride=1, padding=0):

    c_out, c_in, kh, kw = w.shape

    cols, h_out, w_out = im2col(x, kh, kw, stride, padding)

    w_flat = w.reshape(c_out, -1)

    out = w_flat @ cols

    if b is not None:

        out += b[:, None]

    return out.reshape(c_out, h_out, w_out)

In [ ]:
```

Correctness check: run both implementations and compare.

In [ ]:
```python

rng = np.random.default_rng(0)

x = rng.normal(0, 1, (3, 16, 16)).astype(np.float32)

w = rng.normal(0, 1, (8, 3, 3, 3)).astype(np.float32)

b = rng.normal(0, 1, (8,)).astype(np.float32)

y_naive = conv2d_naive(x, w, b, padding=1)

y_im2col = conv2d_im2col(x, w, b, padding=1)

print(f"max abs diff: {np.max(np.abs(y_naive - y_im2col)):.2e}")

In [ ]:
```

`max abs diff` should be around `1e-5` — the difference is floating-point accumulation order, not a bug.

### Step 6: A bank of hand-designed kernels

Five filters that show what a single conv layer can express before any training.

In [ ]:
```python

KERNELS = {

    "identity": np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=np.float32),

    "blur_3x3": np.ones((3, 3), dtype=np.float32) / 9.0,

    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),

    "sobel_x": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),

    "sobel_y": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32),

}

def apply_kernel(img2d, kernel):

    x = img2d[None].astype(np.float32)

    w = kernel[None, None]

    return conv2d_im2col(x, w, padding=1)[0]

In [ ]:
```

Applied to any grayscale image, blur softens, sharpen crisps up edges, Sobel-x lights up vertical edges, Sobel-y lights up horizontal edges. These are exactly the patterns that the *first* trained conv layer in AlexNet and VGG ended up learning — because a good image model needs edge and blob detectors no matter what task comes later.

## Exercises

In [ ]:
1. **(Easy)** Given a 128x128 grayscale input and a stack of `[Conv3x3(s=1,p=1), Conv3x3(s=2,p=1), Conv3x3(s=1,p=1), Conv3x3(s=2,p=1)]`, compute the output spatial size and the receptive field at each layer by hand. Verify with a PyTorch `nn.Sequential` of dummy convs.
2. **(Medium)** Extend `conv2d_naive` and `conv2d_im2col` to accept a `groups` argument. Show that `groups=C_in=C_out` reproduces a depthwise convolution and that its parameter count is `C * K * K` instead of `C * C * K * K`.
3. **(Hard)** Implement the backward pass of `conv2d_im2col` by hand: given the gradient of the output, compute the gradient of `x` and `w`. Verify against `torch.autograd.grad` on the same inputs and weights. The trick: the gradient of im2col is `col2im`, and it has to accumulate overlapping windows.